In [14]:
# pip install boto3

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import monotonically_increasing_id

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("fact_transactions") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.profile",
        "default"
    ) \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [16]:
spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.credentials.provider",
    "com.amazonaws.auth.profile.ProfileCredentialsProvider"
)

spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.profile",
    "default"
)

In [17]:
transactions = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/transactions/"
)

dim_currency = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/dimensional_model/dim_currency/"
)

In [18]:
transactions.show(5)
dim_currency.show(5)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+
|2022-09-01 00:16:00|        1|   8000EC1E0|      1| 8000EC1E0|          11.86|         US Dollar|      11.86|       US Dollar|  Reinvestment|            0|         16|           1|2022|    9|
|2022-09-01 00:04:00|        1|   8000F4510|  11813| 8011305D0|           9.82|         US Dollar|       9.82|       US Dollar|   Credit Card|            0|          4|           1|2022|    9|
|2022-09-01 00:11:00|       12|   8

In [19]:
transactions.printSchema()
dim_currency.printSchema()

root
 |-- Timestamp: timestamp (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account_From: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account_To: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)
 |-- minute_part: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

root
 |-- currency: string (nullable = true)
 |-- currency_id: long (nullable = true)



In [20]:
df_result = transactions.join(
    dim_currency,
    transactions["Payment Currency"] == dim_currency["currency"],
    "left"
).withColumnRenamed("currency_id", "payment_currency_id") \
 .drop("currency")

df_result.show(5)   

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|payment_currency_id|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+
|2022-09-01 00:16:00|        1|   8000EC1E0|      1| 8000EC1E0|          11.86|         US Dollar|      11.86|       US Dollar|  Reinvestment|            0|         16|           1|2022|    9|                  0|
|2022-09-01 00:04:00|        1|   8000F4510|  11813| 8011305D0|           9.82|         US Dollar|       9.82|       US Dollar|   Credit Card|      

In [21]:
df_result = df_result.join(
    dim_currency,
    df_result["Receiving Currency"] == dim_currency["currency"],
    "left"
).withColumnRenamed("currency_id", "receiving_currency_id") \
 .drop("currency")

df_result.show(5)  

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+---------------------+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|payment_currency_id|receiving_currency_id|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+---------------------+
|2022-09-01 00:16:00|        1|   8000EC1E0|      1| 8000EC1E0|          11.86|         US Dollar|      11.86|       US Dollar|  Reinvestment|            0|         16|           1|2022|    9|                  0|                    0|
|2022-09-01 00:04:00|        1|   8000F4510|  11813| 8011305

In [22]:
dim_payment_format = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/dimensional_model/dim_payment_format/"
)

df_result = df_result.join(
    dim_payment_format,
    df_result["Payment Format"] == dim_payment_format["Payment"],
    "left"
).withColumnRenamed("Payment_ID", "payment_format_id") \
 .drop("Payment")

In [23]:
df_result.show(5)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+---------------------+-----------------+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|payment_currency_id|receiving_currency_id|payment_format_id|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+---------------------+-----------------+
|2022-09-01 00:16:00|        1|   8000EC1E0|      1| 8000EC1E0|          11.86|         US Dollar|      11.86|       US Dollar|  Reinvestment|            0|         16|           1|2022|    9|                  0|                    0|       

In [24]:
from pyspark.sql import functions as F


dim_date = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/dimensional_model/dim_date/"
)

df_result = df_result.withColumn("fecha_solo", F.to_date("Timestamp"))

df_result = df_result.join(
    dim_date.select("fecha", "day_of_week", "day_of_week_name"),
    df_result["fecha_solo"] == dim_date["fecha"],
    "left"
).drop("fecha_solo")

df_result.show(5)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+---------------------+-----------------+----------+-----------+----------------+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|payment_currency_id|receiving_currency_id|payment_format_id|     fecha|day_of_week|day_of_week_name|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+-------------------+---------------------+-----------------+----------+-----------+----------------+
|2022-09-01 00:16:00|        1|   8000EC1E0|      1| 8000EC1E0|          11.86|         US Dollar|      11.86|       US D

In [25]:
fact_transactions = df_result.select(
    "Timestamp", "fecha", "year", "month", "day_of_month", "day_of_week", "day_of_week_name",
    "Account_From", "Account_To", "From Bank", "To Bank",
    "Amount Paid", "Amount Received",
    "payment_currency_id", "receiving_currency_id", "payment_format_id",
    "Is Laundering"
)

fact_transactions.show(5)

+-------------------+----------+----+-----+------------+-----------+----------------+------------+----------+---------+-------+-----------+---------------+-------------------+---------------------+-----------------+-------------+
|          Timestamp|     fecha|year|month|day_of_month|day_of_week|day_of_week_name|Account_From|Account_To|From Bank|To Bank|Amount Paid|Amount Received|payment_currency_id|receiving_currency_id|payment_format_id|Is Laundering|
+-------------------+----------+----+-----+------------+-----------+----------------+------------+----------+---------+-------+-----------+---------------+-------------------+---------------------+-----------------+-------------+
|2022-09-01 00:16:00|2022-09-01|2022|    9|           1|          5|        Thursday|   8000EC1E0| 8000EC1E0|        1|      1|      11.86|          11.86|                  0|                    0|                5|            0|
|2022-09-01 00:04:00|2022-09-01|2022|    9|           1|          5|        Thur

In [26]:
fact_transactions.count()

5078336

In [27]:
fact_transactions.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ["payment_currency_id", "receiving_currency_id", "payment_format_id", "fecha"]]).show()

+-------------------+---------------------+-----------------+-----+
|payment_currency_id|receiving_currency_id|payment_format_id|fecha|
+-------------------+---------------------+-----------------+-----+
|                  0|                    0|                0|    0|
+-------------------+---------------------+-----------------+-----+



In [28]:
fact_transactions.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("s3a://datapath-buckets/bank_datasets/curated/dimensional_model/fact_transactions/")